In [ ]:
!nvidia-smi

Wed Nov 12 03:52:44 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Clone ProtoViT repo
!git clone https://github.com/Henrymachiyu/ProtoViT.git proto_vits
import sys
sys.path.append('/content/proto_vits')

Cloning into 'proto_vits'...
remote: Enumerating objects: 226, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 226 (delta 58), reused 22 (delta 21), pack-reused 144 (from 1)
Receiving objects: 100% (226/226), 996.51 KiB | 49.83 MiB/s, done.
Resolving deltas: 100% (117/117), done.


In [ ]:
# Install dependencies
!pip install -U pip setuptools wheel
!pip install -r proto_vits/requirements.txt
!pip install timm==0.4.12 augmentor opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


ERROR: Invalid requirement: '_libgcc_mutex=0.1=main': Expected package name at the start of dependency specifier
    _libgcc_mutex=0.1=main
    ^ (from line 4 of proto_vits/requirements.txt)
Hint: = is not a valid operator. Did you mean == ?
  Attempting uninstall: timm
    Found existing installation: timm 1.0.22
    Uninstalling timm-1.0.22:
      Successfully uninstalled timm-1.0.22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [timm]


In [ ]:
# Download CIFAR-10 and create in-memory dataset
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Transform to 224x224 for ViT
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
])

train_ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_ds  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Optional: smaller subset for fast Colab smoke test
#train_ds = Subset(train_ds, range(1000))
#test_ds  = Subset(test_ds, range(200))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

100%|██████████| 170M/170M [00:14<00:00, 11.8MB/s]


In [ ]:
# Update ProtoViT settings dynamically
import types
import settings
# settings = types.SimpleNamespace()
# settings.train_dir = ''  # Not used
# settings.test_dir  = ''  # Not used
# settings.train_push_dir = ''  # Not used
# settings.num_classes = 10
# settings.img_size = 224
# settings.base_architecture = 'resnet34'  # or use default in repo
# settings.prototype_shape = [10, 192, 1, 1]  # small for smoke test
# settings.prototype_activation_function = 'log'
# settings.add_on_layers_type = 'linear'
# settings.sig_temp = 1.0
# settings.radius = 1.0

In [ ]:
# Import model and construct Proto-ViT
import model
ppnet = model.construct_PPNet(base_architecture=settings.base_architecture,
                               pretrained=True,
                               img_size=settings.img_size,
                               prototype_shape=settings.prototype_shape,
                               num_classes=settings.num_classes,
                               prototype_activation_function=settings.prototype_activation_function,
                               sig_temp=settings.sig_temp,
                               radius=settings.radius,
                               add_on_layers_type=settings.add_on_layers_type)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ppnet = ppnet.to(device)

https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth


In [ ]:
# Quick forward pass to test
dummy = torch.randn(2, 3, 224, 224).to(device)
out = ppnet(dummy)
print(out)
#print('Output shape:', out.shape)

(tensor([[60.4386, 60.1597, 60.5280, 59.8423, 60.0302, 59.7871, 59.9836, 59.9582,
         60.1743, 59.7649, 60.0536, 59.9823, 59.9787, 60.3183, 60.0230, 60.3909,
         60.1870, 60.2935, 59.6771, 60.2348, 59.8311, 60.3538, 60.0669, 59.9770,
         60.1553, 60.0512, 59.7679, 60.0923, 59.8638, 60.3100, 60.2815, 60.3182,
         60.4196, 60.1163, 60.0503, 60.1060, 60.3370, 60.2377, 59.9521, 59.9637,
         60.1096, 60.3428, 60.2579, 60.0355, 59.7761, 60.0510, 60.1265, 59.9916,
         59.9944, 60.1859, 60.1334, 59.9912, 60.1312, 60.0786, 60.0130, 60.0186,
         59.8825, 60.0430, 60.0167, 60.0053, 60.2925, 60.1220, 60.0293, 60.2938,
         60.5424, 59.7454, 59.8303, 60.0320, 60.2171, 59.8200, 60.1485, 59.7199,
         60.0276, 60.1321, 60.3129, 60.3533, 60.2134, 59.9744, 60.3816, 60.0339,
         60.2922, 60.0535, 59.8531, 60.2629, 60.2700, 59.8500, 59.9708, 60.3597,
         60.1266, 60.2317, 60.5033, 60.4208, 60.3777, 59.8065, 60.0310, 60.1481,
         60.2748, 60.2359, 

/content/proto_vits/model.py:207: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  start_row = torch.tensor(center_row+self.radius - small_size // 2)
/content/proto_vits/model.py:208: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  start_col = torch.tensor(center_col+self.radius - small_size // 2)


In [ ]:
%cd proto_vits

[Errno 2] No such file or directory: 'proto_vits'
/content/proto_vits


In [ ]:
# Minimal training loop for smoke test
import train_and_test as tnt
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# --- Optimizer ---
optimizer = AdamW(ppnet.parameters(), lr=2e-4, weight_decay=2e-4)

# --- OneCycle schedule (helps avoid early collapse) ---
scheduler = OneCycleLR(
    optimizer,
    max_lr=2e-4,
    epochs=80,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,  # slow warm-up
    div_factor=25,  # very low initial LR
    final_div_factor=1e4
)

# --- Regularization coefficients (stabilized) ---
coefs = {
    'crs_ent': 1.0,
    'clst': 0.6,
    'sep': 0.6,
    'l1': 1e-4,
    'orth': 5e-3,
    'coh': 5e-3
}

# --- Epochs ---
num_epochs = 30

for epoch in range(num_epochs):
    print('Epoch', epoch)
    _, train_loss = tnt.train(model=ppnet, dataloader=train_loader, optimizer=optimizer,
                               class_specific=True, coefs=coefs, log=print, ema=None, clst_k=1, sum_cls=1)
    accu, tst_loss = tnt.test(model=ppnet, dataloader=test_loader,
                               class_specific=True, log=print, clst_k=1, sum_cls=1)
    print('Test Accuracy:', accu)

Epoch 0
	train
	time: 	1056.394379377365
	total loss: 	321.6391854606526
	cross ent: 	1.3045098404623496
	orthogonal loss	59420.120948796386
	cluster: 	0.4562868654766071
	slot of prototype 0: 	tensor([1.0000, 0.9999, 0.9999, 0.9999], device='cuda:0',
       grad_fn=<SelectBackward0>)
	Estimated avg number of subpatches: 	4.0
	Estimated avg slots logit: 	tensor([0.0020, 0.0020, 0.0020,  ..., 0.0020, 0.0020, 0.0020], device='cuda:0',
       grad_fn=<DivBackward0>)
	separation:	0.40592703388161333
	avg separation:	0.014616808217520554
	coherence loss: 		537.713407608003%
	accu: 		78.474%
	l1: 		202520.796875
	p dist pair: 	271.38909912109375
	test
	time: 	141.50034475326538
	total loss: 	0.0
	cross ent: 	0.3229683470040465
	orthogonal loss	58821.609375
	cluster: 	0.5048866443359814
	slot of prototype 0: 	tensor([1.0000, 0.9999, 0.9999, 0.9999], device='cuda:0')
	Estimated avg number of subpatches: 	4.0
	Estimated avg slots logit: 	tensor([0.0020, 0.0020, 0.0020,  ..., 0.0020, 0.0020, 0.0

In [ ]:
torch.save(ppnet.state_dict(), 'ppnet_epoch30.pth')

In [ ]:
!python /content/proto_vits/main.py

In [ ]:
img_size = 224


# Data loader for prototype pushing (no normalization)
push_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),  # keeps range [0,1]
])

push_train_ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=push_transform)
push_train_ds = Subset(push_train_ds, range(1000))

push_train_loader = DataLoader(push_train_ds, batch_size=32, shuffle=True)

In [ ]:
import push_greedy
from preprocess import preprocess_input_function
import os

# Directory to save prototype visuals
img_dir = './proto_visuals'
os.makedirs(img_dir, exist_ok=True)

ppnet.load_state_dict(torch.load('ppnet_epoch30.pth'))
ppnet.eval()

push_greedy.push_prototypes(
    dataloader=push_train_loader,        # can reuse your CIFAR loader
    pnet=ppnet,
    class_specific=True,
    preprocess_input_function=preprocess_input_function,
    prototype_layer_stride=1,
    root_dir_for_saving_prototypes=img_dir,
    epoch_number=30,
    prototype_img_filename_prefix='proto-img',
    prototype_self_act_filename_prefix='proto-self-act',
    proto_bound_boxes_filename_prefix='proto-bbox',
    save_prototype_class_identity=True,
    log=print,
)


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

In [ ]:
import matplotlib.pyplot as plt
import glob
from PIL import Image


imgs = sorted(glob.glob('./proto_visuals/epoch-0/proto-img-*.png'))[:16]
print(len(imgs))
plt.figure(figsize=(10,10))
for i, path in enumerate(imgs):
    plt.subplot(4,4,i+1)
    plt.imshow(Image.open(path))
    plt.axis('off')
plt.suptitle('Top-Activating Prototype Patches')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import glob
from PIL import Image


imgs = sorted(glob.glob('./proto_visuals/epoch-30/proto-img-*.png'))[:16]
print(len(imgs))
plt.figure(figsize=(10,10))
for i, path in enumerate(imgs):
    plt.subplot(4,4,i+1)
    plt.imshow(Image.open(path))
    plt.axis('off')
plt.suptitle('Top-Activating Prototype Patches')
plt.show()


In [ ]:
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image

# Load a test image
img_path = "bird_2.jpg"
img = Image.open(img_path).convert("RGB")

# same preprocess used in training
x = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
])(img).unsqueeze(0).to(device)

# Get model output + prototype activations
with torch.no_grad():
    conv_features, min_distances, patch_indices = ppnet.push_forward(x)
    topk = torch.topk(-min_distances.squeeze(), k=5)
    proto_ids = topk.indices.cpu().numpy()
    print("Top prototypes:", proto_ids)
    # logits, proto_acts = ppnet.forward_with_proto_activations(x)
    # pred = torch.argmax(logits, dim=1).item()

#print(f"Predicted class: {pred}")

# Visualize strongest prototypes
#ppnet.visualize_activations(x, proto_acts, top_k=5)


In [ ]:
bbox_imgs = sorted(['./proto_visuals/epoch-30/proto-imgbbox-original' + str(i) + '.png' for i in proto_ids])
plt.figure(figsize=(10,10))
for j, path in enumerate(bbox_imgs):
    plt.subplot(4,4,j+1)
    plt.imshow(Image.open(path))
    plt.axis('off')
plt.suptitle('Prototypes with Bounding Boxes')
plt.show()


In [ ]:
# import re

# with open('/content/proto_vits/push_greedy.py', 'r') as f:
#     push_greedy_code = f.read()

# # Add clipping to the image data before saving
# modified_push_greedy_code = re.sub(
#     r"plt\.imsave\((.*?),\s*original_img_j,\s*vmin=0\.0,",
#     r"plt.imsave(\1, original_img_j.clip(0, 1), vmin=0.0,",
#     push_greedy_code,
#     count=1
# )

# with open('/content/proto_vits/push_greedy.py', 'w') as f:
#     f.write(modified_push_greedy_code)

# print("Modified push_greedy.py to clip image values before saving.")